# LangGraph Conditional + Iterative Workflows

**VidTrace course reconstruction — Agentic AI Class, 4 Sep 2026**

The extracted lecture covers conditional and iterative workflows. The source material shows:
- **LLM-Based Conditional Workflow: Sentiment Review Flow**
- State `review`, `sentiment`, `diagnosis`, `response`
- `Find Sentiment (LLM)`
- `Check Sentiment`
- Positive / Negative branches
- `Run Diagnosis (LLM)`

The same extraction also shows an **Iterative Workflow: Tweet Refinement Loop** with Generate Tweet, Evaluate Tweet, Optimize Tweet and a maximum-iteration check.

In [ ]:
!pip -q install -U langgraph langchain-core

from typing import TypedDict, Literal, Any
from pprint import pprint
from langgraph.graph import StateGraph, START, END

print("LangGraph setup complete.")

## 1. Conditional Workflow — Sentiment Review Flow

The condition is evaluated after sentiment detection. The routing function chooses one downstream path.

In [ ]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"] | None
    diagnosis: dict[str, Any] | None
    response: str | None

def find_sentiment(state: ReviewState):
    text = state["review"].lower()
    negative_words = ["bad", "poor", "terrible", "slow", "disappointed"]
    sentiment = "negative" if any(w in text for w in negative_words) else "positive"
    return {"sentiment": sentiment}

def route_sentiment(state: ReviewState) -> str:
    if state["sentiment"] == "positive":
        return "positive_response"
    return "run_diagnosis"

def positive_response(state: ReviewState):
    return {
        "response": "Thank you for the positive feedback!"
    }

def run_diagnosis(state: ReviewState):
    return {
        "diagnosis": {
            "issue_type": "service",
            "tone": "negative",
            "urgency": "medium"
        }
    }

def negative_response(state: ReviewState):
    return {
        "response": "We are sorry about the experience. Your concern has been noted."
    }

r = StateGraph(ReviewState)
r.add_node("find_sentiment", find_sentiment)
r.add_node("positive_response", positive_response)
r.add_node("run_diagnosis", run_diagnosis)
r.add_node("negative_response", negative_response)

r.add_edge(START, "find_sentiment")
r.add_conditional_edges(
    "find_sentiment",
    route_sentiment,
    {
        "positive_response": "positive_response",
        "run_diagnosis": "run_diagnosis"
    }
)
r.add_edge("run_diagnosis", "negative_response")
r.add_edge("positive_response", END)
r.add_edge("negative_response", END)

review_graph = r.compile()

positive_result = review_graph.invoke({
    "review": "The service was excellent and fast.",
    "sentiment": None,
    "diagnosis": None,
    "response": None
})

negative_result = review_graph.invoke({
    "review": "The service was slow and disappointing.",
    "sentiment": None,
    "diagnosis": None,
    "response": None
})

print("Positive path:")
pprint(positive_result)
print("\nNegative path:")
pprint(negative_result)

### What is conditional here?

The graph does not blindly follow one fixed path. After `find_sentiment`, the routing function inspects the state and chooses the next node.

This is the core conditional-edge idea shown in the lecture.

## 2. Iterative Workflow — Tweet Refinement Loop

The extracted slide shows the loop:

`Start → Generate Tweet → Evaluate Tweet → Approved? / Needs Improvement → Optimize Tweet → Max Iteration? → End`

When the tweet is not approved, the workflow loops back through optimization and evaluation.

In [ ]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    feedback: str
    approved: bool
    iteration: int
    max_iterations: int

def generate_tweet(state: TweetState):
    tweet = f"Learning about {state['topic']} #AI #LangGraph"
    return {
        "tweet": tweet,
        "feedback": "",
        "approved": False,
        "iteration": 0
    }

def evaluate_tweet(state: TweetState):
    tweet = state["tweet"]
    approved = len(tweet) <= 80 and "#LangGraph" in tweet
    feedback = "Approved" if approved else "Needs improvement"
    return {"approved": approved, "feedback": feedback}

def route_after_evaluation(state: TweetState) -> str:
    return "end" if state["approved"] else "optimize_tweet"

def optimize_tweet(state: TweetState):
    optimized = state["tweet"][:70].rstrip() + " #AgenticAI"
    return {
        "tweet": optimized,
        "iteration": state["iteration"] + 1
    }

def route_after_optimization(state: TweetState) -> str:
    if state["iteration"] >= state["max_iterations"]:
        return "end"
    return "evaluate_tweet"

t = StateGraph(TweetState)
t.add_node("generate_tweet", generate_tweet)
t.add_node("evaluate_tweet", evaluate_tweet)
t.add_node("optimize_tweet", optimize_tweet)

t.add_edge(START, "generate_tweet")
t.add_edge("generate_tweet", "evaluate_tweet")

t.add_conditional_edges(
    "evaluate_tweet",
    route_after_evaluation,
    {
        "end": END,
        "optimize_tweet": "optimize_tweet"
    }
)

t.add_conditional_edges(
    "optimize_tweet",
    route_after_optimization,
    {
        "end": END,
        "evaluate_tweet": "evaluate_tweet"
    }
)

tweet_graph = t.compile()

tweet_result = tweet_graph.invoke({
    "topic": "agentic workflows",
    "tweet": "",
    "feedback": "",
    "approved": False,
    "iteration": 0,
    "max_iterations": 3
})

pprint(tweet_result)

## 3. What to observe

### Conditional workflow
A state value determines which branch runs.

### Iterative workflow
The graph contains a loop. The evaluator can send execution back to an optimization node, and the maximum-iteration condition prevents an unbounded loop.

The lecture's extracted tweet diagram explicitly contains **Generate Tweet → Evaluate Tweet → Approved? / Needs Improvement → Optimize Tweet → Max Iteration? → End**.

In [ ]:
print("Final tweet:", tweet_result["tweet"])
print("Iterations:", tweet_result["iteration"])
print("Approved:", tweet_result["approved"])